In [19]:
import pandas as pd
import os

data_folder = "../cleaned_data"

customers_clean = pd.read_csv(os.path.join(data_folder, "customers_clean.csv"))
orders_clean = pd.read_csv(os.path.join(data_folder, "orders_clean.csv"))
order_items_clean = pd.read_csv(os.path.join(data_folder, "order_items_clean.csv"))
payments_clean = pd.read_csv(os.path.join(data_folder, "payments_clean.csv"))
reviews = pd.read_csv(os.path.join(data_folder, "reviews_clean.csv"))
products_clean = pd.read_csv(os.path.join(data_folder, "products_clean.csv"))
sellers_clean = pd.read_csv(os.path.join(data_folder, "sellers_clean.csv"))
geolocation_clean = pd.read_csv(os.path.join(data_folder, "geolocation_clean.csv"))
category_translation = pd.read_csv(os.path.join(data_folder, "category_translation_clean.csv"))
payment_summary = pd.read_csv(os.path.join(data_folder, "payment_summary.csv"))

print("All cleaned datasets loaded successfully.")

All cleaned datasets loaded successfully.


In [20]:
dataset_check = {
    "Customers": customers_clean,
    "Orders": orders_clean,
    "Order Items": order_items_clean,
    "Payments": payments_clean,
    "Reviews": reviews,
    "Products": products_clean,
    "Sellers": sellers_clean,
    "Geolocation": geolocation_clean,
    "Category Translation": category_translation,
    "Payment Summary": payment_summary
}

for name, data in dataset_check.items():
    print(f"{name}: {data.shape}")

Customers: (99441, 5)
Orders: (99441, 12)
Order Items: (110697, 9)
Payments: (94532, 5)
Reviews: (99224, 7)
Products: (32340, 9)
Sellers: (3095, 3)
Geolocation: (19015, 3)
Category Translation: (71, 2)
Payment Summary: (90779, 3)


In [21]:
date_columns = {
    "orders_clean": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items_clean": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"]
}

dataframes = {
    "orders_clean": orders_clean,
    "order_items_clean": order_items_clean,
    "reviews": reviews
}

for name, columns in date_columns.items():
    for column in columns:
        dataframes[name][column] = pd.to_datetime(
            dataframes[name][column],
            errors="coerce"
        )

print("Date columns converted successfully.")
print(orders_clean[date_columns["orders_clean"]].dtypes)
print("\nOrder item date:")
print(order_items_clean["shipping_limit_date"].dtype)
print("\nReview dates:")
print(reviews[date_columns["reviews"]].dtypes)

Date columns converted successfully.
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Order item date:
datetime64[us]

Review dates:
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [22]:
review_summary = (
    reviews.groupby("order_id", as_index=False)
    .agg(
        avg_review_score=("review_score", "mean"),
        review_count=("review_id", "count"),
        avg_review_response_days=(
            "review_response_days",
            "mean"
        )
    )
)

print("Review Summary Shape:", review_summary.shape)
print(
    "Duplicate order IDs:",
    review_summary["order_id"].duplicated().sum()
)
print(review_summary.head())

Review Summary Shape: (98673, 4)
Duplicate order IDs: 0
                           order_id  avg_review_score  review_count  \
0  00010242fe8c5a6d1ba2dd792cb16214               5.0             1   
1  00018f77f2f0320c557190d7a144bdd3               4.0             1   
2  000229ec398224ef6ca0657da4fc703e               5.0             1   
3  00024acbcdf0a6daa1e931b038114c75               4.0             1   
4  00042b26cf59d7ce69dfabb4e55b4fd9               5.0             1   

   avg_review_response_days  
0                  1.456285  
1                  2.482095  
2                  0.671192  
3                  0.693762  
4                  1.454850  


In [23]:
master_df = order_items_clean.merge(
    orders_clean,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Original order items:", len(order_items_clean))
print("After merge:", len(master_df))
print("\nMerge results:")
print(master_df["_merge"].value_counts())

Original order items: 110697
After merge: 110697

Merge results:
_merge
both          110697
left_only          0
right_only         0
Name: count, dtype: int64


In [24]:
master_df = master_df.drop(columns="_merge")

master_df = master_df.merge(
    customers_clean,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Rows after customer merge:", len(master_df))
print("\nMerge results:")
print(master_df["_merge"].value_counts())

Rows after customer merge: 110697

Merge results:
_merge
both          110697
left_only          0
right_only         0
Name: count, dtype: int64


In [25]:
master_df = master_df.drop(columns="_merge")

master_df = master_df.merge(
    products_clean,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Rows after product merge:", len(master_df))
print("\nMerge results:")
print(master_df["_merge"].value_counts())

Rows after product merge: 110697

Merge results:
_merge
both          109125
left_only       1572
right_only         0
Name: count, dtype: int64


In [26]:
unmatched_products = master_df.loc[
    master_df["product_category_name_english"].isna(),
    ["product_id", "product_category_name_english"]
]

print("Unmatched item rows:", len(unmatched_products))
print("Unique unmatched product IDs:", unmatched_products["product_id"].nunique())
print(unmatched_products.head())

Unmatched item rows: 1572
Unique unmatched product IDs: 580
                           product_id product_category_name_english
123  ff6caf9340512b8bf6d2a2a6df032cfa                           NaN
125  a9c404971d1a5b1cbc2e4070e02731fd                           NaN
132  5a848e4ab52fd5445cdc07aab1c40e48                           NaN
169  e10758160da97891c2fdcbc35f0f031d                           NaN
364  76d1a1a9d21ab677a61c3ae34b1b352f                           NaN


In [27]:
master_df["product_category_name_english"] = (
    master_df["product_category_name_english"]
    .fillna("product_details_unavailable")
)

print(
    master_df["product_category_name_english"]
    .value_counts()
    .head()
)

print(
    "Missing product categories:",
    master_df["product_category_name_english"].isna().sum()
)

product_category_name_english
bed_bath_table           10251
health_beauty             9060
sports_leisure            8018
furniture_decor           7640
computers_accessories     7353
Name: count, dtype: int64
Missing product categories: 0


In [28]:
master_df = master_df.drop(columns="_merge")

master_df = master_df.merge(
    sellers_clean,
    on="seller_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Rows after seller merge:", len(master_df))
print("\nMerge results:")
print(master_df["_merge"].value_counts())

Rows after seller merge: 110697

Merge results:
_merge
both          110697
left_only          0
right_only         0
Name: count, dtype: int64


In [29]:
master_df = master_df.drop(columns="_merge")

master_df = master_df.merge(
    payment_summary,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Rows after payment merge:", len(master_df))
print("\nMerge results:")
print(master_df["_merge"].value_counts())

Rows after payment merge: 110697

Merge results:
_merge
both          101101
left_only       9596
right_only         0
Name: count, dtype: int64


In [30]:
unmatched_payments = master_df.loc[
    master_df["_merge"] == "left_only",
    ["order_id", "total_payment_value", "payment_count"]
]

print("Unmatched item rows:", len(unmatched_payments))
print("Unique unmatched orders:", unmatched_payments["order_id"].nunique())
print("\nMissing payment values:")
print(unmatched_payments[["total_payment_value", "payment_count"]].isna().sum())

print("\nSample unmatched orders:")
print(unmatched_payments.head())

Unmatched item rows: 9596
Unique unmatched orders: 8451

Missing payment values:
total_payment_value    9596
payment_count          9596
dtype: int64

Sample unmatched orders:
                            order_id  total_payment_value  payment_count
17  000aed2e25dbad2f9ddb70584c5a2ded                  NaN            NaN
21  000e906b789b55f64edcb1f84030f90d                  NaN            NaN
30  00137e170939bba5a3134e2386413108                  NaN            NaN
32  00143d0f86d6fbd9f9b38ab440ac16f5                  NaN            NaN
33  00143d0f86d6fbd9f9b38ab440ac16f5                  NaN            NaN


In [31]:
unmatched_order_ids = unmatched_payments["order_id"].unique()

payment_check = payments_clean[
    payments_clean["order_id"].isin(unmatched_order_ids)
]

print("Unmatched orders:", len(unmatched_order_ids))
print("Matching rows in payments_clean:", len(payment_check))
print(
    "Unique orders found in payments_clean:",
    payment_check["order_id"].nunique()
)

print("\nSample payment records:")
print(payment_check.head())

Unmatched orders: 8451
Matching rows in payments_clean: 0
Unique orders found in payments_clean: 0

Sample payment records:
Empty DataFrame
Columns: [order_id, payment_sequential, payment_type, payment_installments, payment_value]
Index: []


In [32]:
# Remove the temporary merge indicator
master_df = master_df.drop(columns=["_merge"])

# Merge review summary
master_df = master_df.merge(
    review_summary,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Rows after review merge:", len(master_df))

print("\nReview merge results:")
print(master_df["_merge"].value_counts())

print("\nMissing review scores:", master_df["avg_review_score"].isna().sum())

Rows after review merge: 110697

Review merge results:
_merge
both          109774
left_only        923
right_only         0
Name: count, dtype: int64

Missing review scores: 923


# Remove temporary merge indicator

In [33]:
master_df = master_df.drop(columns=["_merge"])

print("Master dataset shape:", master_df.shape)

print("\nColumn names:")
print(master_df.columns.tolist())

print("\nFirst 5 rows:")
display(master_df.head())

Master dataset shape: (110697, 39)

Column names:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'item_total_value', 'freight_percentage', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'approval_delay_hours', 'delivery_difference_days', 'delivery_status', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_city', 'seller_state', 'total_payment_value', 'payment_count', 'avg_review_score', 'review_count', 'avg_review_response_days']

First 5 rows:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,item_total_value,freight_percentage,customer_id,...,product_height_cm,product_width_cm,product_category_name_english,seller_city,seller_state,total_payment_value,payment_count,avg_review_score,review_count,avg_review_response_days
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19,22.563667,3ce436f183e68e07877b285a838db11a,...,9.0,14.0,cool_stuff,volta redonda,SP,72.19,1.0,5.0,1.0,1.456285
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83,8.307628,f6dd3ec061db4e3987629fe6b26e5cce,...,30.0,40.0,pet_shop,sao paulo,SP,259.83,1.0,4.0,1.0,2.482095
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87,8.979899,6489ae5e4333f3693df5ad4372dab6d3,...,13.0,33.0,furniture_decor,borda da mata,MG,216.87,1.0,5.0,1.0,0.671192
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78,98.460354,d4eb9395c8c0431ee92fce09860c5a06,...,10.0,15.0,perfumery,franca,SP,25.78,1.0,4.0,1.0,0.693762
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04,9.074537,58dbd0b2d70206bf40e62cd34e84d795,...,40.0,30.0,garden_tools,loanda,PR,218.04,1.0,5.0,1.0,1.454850


In [34]:
# Check whether duplicate review columns exist
review_columns = [
    col for col in master_df.columns
    if "review_score" in col
    or "review_count" in col
    or "review_response_days" in col
]

print("Review-related columns:")
print(review_columns)

print("\nMaster dataset shape:", master_df.shape)

Review-related columns:
['avg_review_score', 'review_count', 'avg_review_response_days']

Master dataset shape: (110697, 39)


In [36]:

# Step 1: Final Master Dataset Validation

print("MASTER DATASET VALIDATION")
print("=" * 40)

# 1. Check dataset dimensions
print("Rows:", master_df.shape[0])
print("Columns:", master_df.shape[1])

# 2. Check duplicate column names
print("\nDuplicate column names:")
print(master_df.columns[master_df.columns.duplicated()].tolist())

# 3. Check fully duplicated rows
print("\nFully duplicated rows:")
print(master_df.duplicated().sum())

# 4. Check duplicate order-item combinations
print("\nDuplicate order-item combinations:")
print(
    master_df.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

# 5. Check missing values
print("\nColumns with missing values:")
missing_values = master_df.isna().sum()
print(missing_values[missing_values > 0].sort_values(ascending=False))

# 6. Check data types
print("\nColumn data types:")
print(master_df.dtypes)


MASTER DATASET VALIDATION
Rows: 110697
Columns: 39

Duplicate column names:
[]

Fully duplicated rows:
0

Duplicate order-item combinations:
0

Columns with missing values:
payment_count                    9596
total_payment_value              9596
delivery_days                    2382
delivery_status                  2375
delivery_difference_days         2375
order_delivered_customer_date    2375
product_photos_qty               1572
product_width_cm                 1572
product_name_length              1572
product_description_length       1572
product_category_name            1572
product_height_cm                1572
product_length_cm                1572
order_delivered_carrier_date     1190
review_count                      923
avg_review_response_days          923
avg_review_score                  923
order_approved_at                  14
approval_delay_hours               14
dtype: int64

Column data types:
order_id                                    str
order_item_id           

In [37]:

# Step 2: Investigate Missing Values

# 1. Missing payments by order status
print("MISSING PAYMENTS BY ORDER STATUS")
print("=" * 40)

print(
    master_df.loc[
        master_df["total_payment_value"].isna(),
        "order_status"
    ].value_counts(dropna=False)
)

# 2. Missing delivery information by order status
print("\nMISSING DELIVERY DATES BY ORDER STATUS")
print("=" * 40)

print(
    master_df.loc[
        master_df["order_delivered_customer_date"].isna(),
        "order_status"
    ].value_counts(dropna=False)
)

# 3. Missing product details
print("\nMISSING PRODUCT DETAILS")
print("=" * 40)

product_columns = [
    "product_category_name",
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print(
    master_df[product_columns]
    .isna()
    .sum()
)

# 4. Missing reviews by order status
print("\nMISSING REVIEWS BY ORDER STATUS")
print("=" * 40)

print(
    master_df.loc[
        master_df["avg_review_score"].isna(),
        "order_status"
    ].value_counts(dropna=False)
)


MISSING PAYMENTS BY ORDER STATUS
order_status
delivered     9392
shipped        108
canceled        35
invoiced        30
processing      29
approved         2
Name: count, dtype: int64

MISSING DELIVERY DATES BY ORDER STATUS
order_status
shipped        1162
canceled        508
invoiced        347
processing      341
delivered         8
unavailable       7
approved          2
Name: count, dtype: int64

MISSING PRODUCT DETAILS
product_category_name         1572
product_name_length           1572
product_description_length    1572
product_photos_qty            1572
product_length_cm             1572
product_height_cm             1572
product_width_cm              1572
dtype: int64

MISSING REVIEWS BY ORDER STATUS
order_status
delivered     811
shipped        83
canceled       17
processing      6
invoiced        6
Name: count, dtype: int64


In [38]:
import os

print("Current notebook directory:", os.getcwd())
print("\nFiles and folders here:")
print(os.listdir("."))

Current notebook directory: C:\Users\ayush\OneDrive\Data analysis project\E-Commerce Customer Intelligence & Business Analytics\notebook

Files and folders here:
['.ipynb_checkpoints', '01_data_understanding.ipynb', '02_data_cleaning.ipynb', '03_Data_Integration_Feature_Engineering.ipynb']


In [39]:
# Step 4: Validate important business rules

print("BUSINESS RULE VALIDATION")
print("=" * 40)

# Check for invalid prices or freight charges
print("Zero or negative prices:", (master_df["price"] <= 0).sum())
print("Negative freight values:", (master_df["freight_value"] < 0).sum())

# Check payment values that exist
print(
    "Negative payment totals:",
    (master_df["total_payment_value"].dropna() < 0).sum()
)

# Check review score range
print(
    "Review scores outside 1–5:",
    (~master_df["avg_review_score"].dropna().between(1, 5)).sum()
)

# Check delivery duration
print(
    "Negative delivery durations:",
    (master_df["delivery_days"].dropna() < 0).sum()
)

# Confirm row count
print("\nFinal row count:", len(master_df))

BUSINESS RULE VALIDATION
Zero or negative prices: 0
Negative freight values: 0
Negative payment totals: 0
Review scores outside 1–5: 0
Negative delivery durations: 0

Final row count: 110697


In [41]:

import os

# Create output folder if it doesn't exist
output_folder = "../cleaned_data"
os.makedirs(output_folder, exist_ok=True)

# Save integrated master dataset
output_path = os.path.join(
    output_folder,
    "olist_master_dataset.csv"
)

master_df.to_csv(output_path, index=False)

print("Master dataset saved successfully!")
print("File path:", os.path.abspath(output_path))
print("Rows:", master_df.shape[0])
print("Columns:", master_df.shape[1])
print("File size (MB):", round(os.path.getsize(output_path) / (1024**2), 2))


Master dataset saved successfully!
File path: C:\Users\ayush\OneDrive\Data analysis project\E-Commerce Customer Intelligence & Business Analytics\cleaned_data\olist_master_dataset.csv
Rows: 110697
Columns: 39
File size (MB): 55.09


In [42]:

# Step 6: Verify the saved master dataset

import pandas as pd

saved_df = pd.read_csv(
    "../cleaned_data/olist_master_dataset.csv"
)

print("Saved dataset verification")
print("=" * 35)
print("Shape:", saved_df.shape)
print("Duplicate rows:", saved_df.duplicated().sum())
print("Duplicate column names:", saved_df.columns.duplicated().sum())
print("File loaded successfully:", saved_df.shape == master_df.shape)


Saved dataset verification
Shape: (110697, 39)
Duplicate rows: 0
Duplicate column names: 0
File loaded successfully: True
